[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/40.%20The%20Port%20Capacity%20%26%20Expansion%20Timing%20Problem/P40-Tier-7_executed.ipynb)

---


# 40. The Port Capacity and Expansion Timing Problem
## Tier 7: Human–AI Partnership (The Centaur Model)

### Goal and workflow

This notebook demonstrates a human–AI decision-support workflow for port capacity planning. The AI component generates and ranks expansion scenarios using a simplified financial and risk model. A simulated human-evaluation stage scores a shortlist against qualitative criteria, after which a rule-based refinement step produces a final recommendation and an explanation.

The workflow is deliberately sequential:

1. Human planners specify strategic parameters and constraints.
2. The AI component samples expansion scenarios and calculates their cost, net present value (NPV), and risk score.
3. Scenarios with the highest AI fitness are sent to a simulated human-evaluation stage.
4. Scenarios that satisfy the modeled human-evaluation criteria are refined using fixed, illustrative feedback rules.
5. A combined AI–human score selects a recommendation, which is reported with its modeled factors.

This is a transparent demonstration of the partnership workflow, not a trained AI system or an empirical study of human decision-making. The evaluation scores are randomly simulated, and the refinement rules are assumptions chosen for illustration. Consequently, the notebook's outputs describe this simulation only; they are not evidence that collaboration improves real-world outcomes.

### Inputs, outputs, and scope

**Inputs** include the planning parameters in `HumanConstraints`, a fixed random seed, and the scenario-generation settings in the code. A scenario specifies an expansion year, capacity increase, technology, and implementation phasing.

**Outputs** include the generated and evaluated scenario lists, the selected recommendation, a textual explanation, and plots that summarize the simulated scenarios.

**Model assumptions and limitations:** Investment cost, annual revenue, discounting, and risk are represented by simplified formulas. Revenue is assumed to be constant after expansion, and operating costs, taxes, inflation, financing structure, construction duration, demand uncertainty, and interactions between projects are not modeled. Environmental impact, labor-transition requirements, community noise limits, and the capacity-buffer requirement are recorded as planning context but are not calculated or enforced because scenarios do not contain the data needed to evaluate them. The financing window is scored in the human-constraint score rather than enforced as a hard feasibility constraint. Scenario generation is random sampling with possible duplicates, not exhaustive enumeration or a guarantee of a global optimum.

### Mathematical structure

Let a scenario be denoted by $s$. Its decision attributes are the expansion year, capacity increase, technology, and whether implementation is phased. The model calculates investment cost $C_s$, NPV $V_s$, and risk score $r_s$ for each generated scenario.

The AI fitness is expressed in dollars so that the NPV and risk penalty use consistent units:

$$
F_{AI}(s) = 10^6 V_s - 10^6 r_s.
$$

Here, $V_s$ is NPV in millions of dollars, so $10^6V_s$ is NPV in dollars. The dimensionless risk score is assigned a penalty of one million dollars per unit of risk. This score ranks scenarios; it is not itself an NPV.

For the simplified financial model, let $t_s$ be the number of years from 2024 to the expansion year, $q_s$ the fractional capacity increase, and $e_s$ the technology-efficiency multiplier. Annual revenue is modeled as $3.2q_s \times 180{,}000e_s$ dollars. At an annual discount rate of $0.08$, the NPV in millions of dollars is:

$$
V_s = \frac{1}{10^6}\sum_{t=t_s}^{14}
\frac{(3.2q_s)(180{,}000)e_s}{(1+0.08)^t} - C_s.
$$

The factor $3.2$ represents the assumed base capacity in millions of TEU; $q_s$ is a fraction, $180{,}000$ is the assumed annual revenue in dollars per additional TEU, and $C_s$ is investment cost in millions of dollars. The calculation treats the investment cost as a present-day amount and assumes the modeled annual revenue begins in the expansion year. These are simplifying assumptions, not a full project-finance model.

The human-constraint score is a weighted rule-based score, denoted $H(s)$. The final selection maximizes the product of AI fitness and this score among the refined scenarios:

$$
s^* = \underset{s \in S_{refined}}{\arg\max}\; F_{AI}(s)H(s).
$$

The product represents the notebook's chosen ranking rule; it is not a probability or a calibrated measure of decision quality. Random human-evaluation attributes are used separately to screen the AI shortlist. The refinement step multiplies NPV by $1.15$ and risk by $0.8$ as fixed illustrative rules. It does not estimate these changes from observed feedback or train a model.

### Computational outline

For $N$ generated scenarios, scenario generation, scoring, evaluation of a fixed-size shortlist, and refinement take linear time in $N$, apart from sorting the scenarios by AI fitness, which takes $O(N\log N)$. The stored scenario lists require $O(N)$ memory. Because scenarios are sampled rather than exhaustively enumerated, these complexity bounds describe the computation, not the coverage of the decision space.

### When this workflow is useful

A human–AI workflow can be useful when a decision combines quantitative analysis with stakeholder concerns and strategic judgment. In practical use, human assessments should come from documented expert input, model assumptions should be validated, and constraints should be enforced only when the required data and operational definitions are available.

In [ ]:
# Import the libraries used for scenario generation, analysis, and visualization.
import random
from dataclasses import dataclass
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Fixed seeds make the simulated scenario and evaluation draws reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

print("Human–AI partnership simulation initialized.")

In [ ]:
@dataclass
class HumanConstraints:
    """Planning parameters and thresholds supplied by human planners."""

    max_investment: float = 500.0  # Million dollars
    environmental_limit: float = 0.1  # Fractional impact limit; not modeled
    labor_transition_months: int = 18
    financing_window_start: int = 2027
    financing_window_end: int = 2029
    capacity_buffer: float = 0.15  # Fractional buffer; not modeled
    community_noise_hours: str = "07:00-19:00"
    stakeholder_acceptance_threshold: float = 0.8


@dataclass
class ExpansionScenario:
    """One candidate expansion plan and its modeled attributes."""

    scenario_id: int
    expansion_year: int
    capacity_increase: float
    technology: str
    phased: bool
    investment_cost: float
    npv: float
    risk_score: float
    technical_feasibility: float = 0.0
    stakeholder_acceptance: float = 0.0
    strategic_alignment: float = 0.0
    implementation_complexity: float = 0.0

    def calculate_ai_fitness(self) -> float:
        """Return a ranking score in dollars, with a $1M-per-risk-unit penalty."""
        npv_dollars = self.npv * 1_000_000
        risk_penalty_dollars = self.risk_score * 1_000_000
        return npv_dollars - risk_penalty_dollars

    def calculate_human_score(self, constraints: HumanConstraints) -> float:
        """Score scenario attributes against the specified planning preferences."""
        score = 0.0

        if self.investment_cost <= constraints.max_investment:
            score += 0.25
        else:
            score -= 0.5

        if constraints.financing_window_start <= self.expansion_year <= constraints.financing_window_end:
            score += 0.2
        else:
            score -= 0.3

        if self.technology == "fully_automated":
            score += 0.15
        elif self.technology == "semi_automated":
            score += 0.1

        if self.phased:
            score += 0.1

        if self.capacity_increase >= 0.25:
            score += 0.15

        if self.risk_score <= 0.3:
            score += 0.15

        return max(0.0, score)


class HumanAIPartnership:
    """Generate, evaluate, refine, and rank port expansion scenarios."""

    def __init__(self, constraints: HumanConstraints):
        self.constraints = constraints
        self.scenarios: List[ExpansionScenario] = []
        self.human_evaluated_scenarios: List[ExpansionScenario] = []
        self.collaborative_scenarios: List[ExpansionScenario] = []
        self.final_recommendation: Optional[ExpansionScenario] = None

    def generate_ai_scenarios(self, num_scenarios: int = 15000) -> List[ExpansionScenario]:
        """Sample scenarios and retain candidates within the initial cost screen."""
        if num_scenarios < 0:
            raise ValueError("num_scenarios must be non-negative.")

        scenarios = []

        for scenario_id in range(num_scenarios):
            scenario = ExpansionScenario(
                scenario_id=scenario_id,
                expansion_year=random.randint(2025, 2030),
                capacity_increase=random.choice([0.15, 0.25, 0.40, 0.60, 0.85]),
                technology=random.choice(
                    ["manual", "semi_automated", "fully_automated"]
                ),
                phased=random.choice([True, False]),
                investment_cost=0.0,
                npv=0.0,
                risk_score=0.0,
            )

            # Costs must be available before NPV and risk can be calculated.
            scenario.investment_cost = self._calculate_investment_cost(scenario)
            scenario.npv = self._calculate_npv(scenario)
            scenario.risk_score = self._calculate_risk_score(scenario)

            # The 20% screening allowance preserves over-budget candidates for scoring.
            if scenario.investment_cost <= self.constraints.max_investment * 1.2:
                scenarios.append(scenario)

        self.scenarios = scenarios
        print(f"AI scenario sampling retained {len(scenarios)} candidates.")
        return scenarios

    def _calculate_investment_cost(self, scenario: ExpansionScenario) -> float:
        """Calculate investment cost in millions of dollars."""
        base_costs = {0.15: 120, 0.25: 200, 0.40: 450, 0.60: 600, 0.85: 750}
        technology_multipliers = {
            "manual": 1.0,
            "semi_automated": 1.2,
            "fully_automated": 1.5,
        }
        phasing_multiplier = 1.15 if scenario.phased else 1.0

        base_cost = base_costs[scenario.capacity_increase]
        technology_multiplier = technology_multipliers[scenario.technology]
        return base_cost * technology_multiplier * phasing_multiplier

    def _calculate_npv(self, scenario: ExpansionScenario) -> float:
        """Calculate simplified NPV in millions of dollars."""
        years_to_expansion = scenario.expansion_year - 2024
        additional_capacity = 3.2 * scenario.capacity_increase
        annual_revenue = additional_capacity * 180_000

        technology_efficiency = {
            "manual": 1.0,
            "semi_automated": 1.15,
            "fully_automated": 1.30,
        }
        efficiency = technology_efficiency[scenario.technology]

        discounted_revenue_dollars = 0.0
        for year in range(years_to_expansion, 15):
            discounted_revenue_dollars += (
                annual_revenue * efficiency / ((1 + 0.08) ** year)
            )

        return discounted_revenue_dollars / 1_000_000 - scenario.investment_cost

    def _calculate_risk_score(self, scenario: ExpansionScenario) -> float:
        """Calculate the modeled risk score, where lower values are preferred."""
        risk = 0.0

        if scenario.investment_cost > 400:
            risk += 0.3
        elif scenario.investment_cost > 200:
            risk += 0.15

        if scenario.technology == "fully_automated":
            risk += 0.2

        if scenario.expansion_year < 2026:
            risk += 0.1

        if scenario.capacity_increase > 0.6:
            risk += 0.15

        return min(risk, 1.0)

    def human_evaluation(
        self, top_scenarios: List[ExpansionScenario]
    ) -> List[ExpansionScenario]:
        """Simulate expert scores and retain candidates meeting both thresholds."""
        evaluated_scenarios = []

        for scenario in top_scenarios:
            # These random values are proxies for human judgments, not observed data.
            scenario.technical_feasibility = random.uniform(0.6, 0.95)
            scenario.stakeholder_acceptance = random.uniform(0.5, 0.9)
            scenario.strategic_alignment = random.uniform(0.7, 0.95)
            scenario.implementation_complexity = random.uniform(0.3, 0.8)

            human_evaluation_score = (
                scenario.technical_feasibility * 0.3
                + scenario.stakeholder_acceptance * 0.3
                + scenario.strategic_alignment * 0.25
                + (1 - scenario.implementation_complexity) * 0.15
            )

            if (
                human_evaluation_score >= 0.7
                and scenario.stakeholder_acceptance
                >= self.constraints.stakeholder_acceptance_threshold
            ):
                evaluated_scenarios.append(scenario)

        self.human_evaluated_scenarios = evaluated_scenarios
        print(
            "Simulated human evaluation approved "
            f"{len(evaluated_scenarios)} of {len(top_scenarios)} shortlisted scenarios."
        )
        return evaluated_scenarios

    def collaborative_refinement(
        self, human_scenarios: List[ExpansionScenario]
    ) -> List[ExpansionScenario]:
        """Apply the notebook's fixed illustrative feedback rules to approved plans."""
        refined_scenarios = [
            self._refine_scenario(scenario) for scenario in human_scenarios
        ]
        self.collaborative_scenarios = refined_scenarios
        return refined_scenarios

    def _refine_scenario(self, scenario: ExpansionScenario) -> ExpansionScenario:
        """Copy a scenario and apply the stated NPV and risk adjustments."""
        return ExpansionScenario(
            scenario_id=scenario.scenario_id,
            expansion_year=scenario.expansion_year,
            capacity_increase=scenario.capacity_increase,
            technology=scenario.technology,
            phased=scenario.phased,
            investment_cost=scenario.investment_cost,
            npv=scenario.npv * 1.15,
            risk_score=scenario.risk_score * 0.8,
            technical_feasibility=scenario.technical_feasibility,
            stakeholder_acceptance=scenario.stakeholder_acceptance,
            strategic_alignment=scenario.strategic_alignment,
            implementation_complexity=scenario.implementation_complexity,
        )

    def generate_final_recommendation(
        self, collaborative_scenarios: List[ExpansionScenario]
    ) -> Optional[ExpansionScenario]:
        """Select the refined scenario with the highest combined ranking score."""
        if not collaborative_scenarios:
            self.final_recommendation = None
            return None

        best_scenario = max(
            collaborative_scenarios,
            key=lambda scenario: scenario.calculate_ai_fitness()
            * scenario.calculate_human_score(self.constraints),
        )
        self.final_recommendation = best_scenario
        return best_scenario

    def explain_decision(self, scenario: ExpansionScenario) -> Dict[str, object]:
        """Summarize modeled attributes without implying unmodeled guarantees."""
        budget_status = (
            "within the stated investment limit"
            if scenario.investment_cost <= self.constraints.max_investment
            else "above the stated investment limit"
        )
        timing_status = (
            "inside the financing window"
            if self.constraints.financing_window_start
            <= scenario.expansion_year
            <= self.constraints.financing_window_end
            else "outside the financing window"
        )

        return {
            "primary_reason": (
                "Highest combined ranking score among the refined candidates; "
                f"modeled NPV is ${scenario.npv:.1f}M and risk score is "
                f"{scenario.risk_score:.1%}."
            ),
            "key_factors": [
                f"Investment of ${scenario.investment_cost:.1f}M is {budget_status}.",
                f"Expansion in {scenario.expansion_year} is {timing_status}.",
                f"Capacity increase is {scenario.capacity_increase:.0%}.",
                f"Technology choice is {scenario.technology.replace('_', ' ')}.",
                f"Implementation is {'phased' if scenario.phased else 'single-phase'}.",
                "Simulated stakeholder acceptance is "
                f"{scenario.stakeholder_acceptance:.1%}.",
            ],
            "modeling_note": (
                "NPV and risk are outputs of simplified assumptions. "
                "The refinement adjustments are illustrative rules, not "
                "empirically estimated effects."
            ),
        }


print("Scenario, evaluation, and recommendation components are ready.")

In [ ]:
# Run the scenario-generation, evaluation, and recommendation workflow.
print("\n" + "=" * 80)
print("HUMAN–AI PARTNERSHIP: PORT CAPACITY PLANNING")
print("=" * 80)

constraints = HumanConstraints()

print("\nPLANNING PARAMETERS")
print(f"Maximum investment: ${constraints.max_investment:.1f}M")
print(
    "Financing window: "
    f"{constraints.financing_window_start}–{constraints.financing_window_end}"
)
print(f"Labor transition notice: {constraints.labor_transition_months} months")
print(f"Capacity buffer parameter: {constraints.capacity_buffer:.0%} (not modeled)")
print(
    "Stakeholder acceptance threshold: "
    f"{constraints.stakeholder_acceptance_threshold:.0%}"
)
print("Environmental limit: recorded but not modeled")
print(f"Community construction hours: {constraints.community_noise_hours} (not modeled)")

partnership = HumanAIPartnership(constraints)

print("\n" + "=" * 60)
print("PHASE 1: AI SCENARIO GENERATION")
print("=" * 60)
ai_scenarios = partnership.generate_ai_scenarios(num_scenarios=15000)

ai_scenarios.sort(key=lambda scenario: scenario.calculate_ai_fitness(), reverse=True)
top_ai_scenarios = ai_scenarios[:50]

print("\nTop five scenarios by AI fitness:")
for rank, scenario in enumerate(top_ai_scenarios[:5], start=1):
    print(
        f"{rank}. Year {scenario.expansion_year}; "
        f"capacity increase {scenario.capacity_increase:.0%}; "
        f"{scenario.technology.replace('_', ' ')}; "
        f"{'phased' if scenario.phased else 'single-phase'}"
    )
    print(
        f"   Investment: ${scenario.investment_cost:.1f}M; "
        f"NPV: ${scenario.npv:.1f}M; risk score: {scenario.risk_score:.1%}"
    )

print("\n" + "=" * 60)
print("PHASE 2: SIMULATED HUMAN EVALUATION")
print("=" * 60)
human_scenarios = partnership.human_evaluation(top_ai_scenarios)

if human_scenarios:
    average_npv = np.mean([scenario.npv for scenario in human_scenarios])
    average_investment = np.mean(
        [scenario.investment_cost for scenario in human_scenarios]
    )
    average_risk = np.mean([scenario.risk_score for scenario in human_scenarios])
    technology_counts = pd.Series(
        [scenario.technology for scenario in human_scenarios]
    ).value_counts().to_dict()

    print("\nApproved-scenario summary:")
    print(f"Average NPV: ${average_npv:.1f}M")
    print(f"Average investment: ${average_investment:.1f}M")
    print(f"Average risk score: {average_risk:.1%}")
    print(f"Technology counts: {technology_counts}")
else:
    print("No shortlisted scenarios met both simulated evaluation thresholds.")

print("\n" + "=" * 60)
print("PHASE 3: COLLABORATIVE REFINEMENT")
print("=" * 60)
collaborative_scenarios = partnership.collaborative_refinement(human_scenarios)

if collaborative_scenarios:
    original_average_npv = np.mean([scenario.npv for scenario in human_scenarios])
    refined_average_npv = np.mean(
        [scenario.npv for scenario in collaborative_scenarios]
    )
    npv_change_percent = (
        (refined_average_npv - original_average_npv) / abs(original_average_npv) * 100
        if original_average_npv != 0
        else float("nan")
    )

    print(f"Refined candidate count: {len(collaborative_scenarios)}")
    print(f"Average NPV before refinement: ${original_average_npv:.1f}M")
    print(f"Average NPV after refinement: ${refined_average_npv:.1f}M")
    print(
        f"Average NPV change under the fixed rule: {npv_change_percent:.1f}%"
    )
    print("This change follows directly from the illustrative 15% NPV adjustment.")
else:
    print("No scenarios are available for refinement.")

print("\n" + "=" * 60)
print("PHASE 4: FINAL RECOMMENDATION")
print("=" * 60)
final_scenario = partnership.generate_final_recommendation(collaborative_scenarios)

if final_scenario is not None:
    print("\nFINAL SIMULATED RECOMMENDATION")
    print(f"Expansion year: {final_scenario.expansion_year}")
    print(f"Capacity increase: {final_scenario.capacity_increase:.0%}")
    print(f"Technology: {final_scenario.technology.replace('_', ' ').title()}")
    print(f"Implementation: {'Phased' if final_scenario.phased else 'Single-phase'}")
    print(f"Investment: ${final_scenario.investment_cost:.1f}M")
    print(f"Modeled NPV: ${final_scenario.npv:.1f}M")
    print(f"Modeled risk score: {final_scenario.risk_score:.1%}")

    explanation = partnership.explain_decision(final_scenario)
    print("\nExplanation")
    print(explanation["primary_reason"])
    for factor in explanation["key_factors"]:
        print(f"- {factor}")
    print(explanation["modeling_note"])
else:
    print("No suitable scenario was selected from the simulated candidates.")

## Reading the workflow results

The scenario list is a sample from the choices encoded in the generator. The top-ranked scenarios are those with the largest AI fitness values, not necessarily those with the largest NPV alone: the fitness score also subtracts a risk penalty. The simulated human-evaluation stage then applies a score threshold and the stated stakeholder-acceptance threshold to that shortlist.

The final recommendation is selected only from the scenarios that pass the simulated evaluation and refinement stages. Its cost, NPV, risk score, and acceptance value are model outputs conditional on the assumptions and random seed in this notebook. In particular, the fixed refinement multipliers create an assumed change in NPV and risk; they do not measure an observed effect of human input. A recommendation that exceeds the investment limit can still remain a candidate because the limit affects its human-constraint score rather than acting as a hard exclusion in the final selection.

In [ ]:
# Plot the screened, evaluated, and refined scenarios when a recommendation exists.
if human_scenarios and collaborative_scenarios and final_scenario is not None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Simulated Human–AI Partnership Results", fontsize=16, fontweight="bold")

    original_npvs = [scenario.npv for scenario in human_scenarios]
    refined_npvs = [scenario.npv for scenario in collaborative_scenarios]

    axes[0, 0].scatter(original_npvs, refined_npvs, alpha=0.65, s=50)
    lower_bound = min(original_npvs + refined_npvs)
    upper_bound = max(original_npvs + refined_npvs)
    axes[0, 0].plot(
        [lower_bound, upper_bound],
        [lower_bound, upper_bound],
        "r--",
        alpha=0.8,
        label="No change",
    )
    axes[0, 0].set_xlabel("NPV before refinement ($M)")
    axes[0, 0].set_ylabel("NPV after refinement ($M)")
    axes[0, 0].set_title("NPV Under the Illustrative Refinement Rule")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    technology_counts = pd.Series(
        [scenario.technology for scenario in collaborative_scenarios]
    ).value_counts()
    bars = axes[0, 1].bar(
        technology_counts.index,
        technology_counts.values,
        color=["skyblue", "lightgreen", "salmon"][: len(technology_counts)],
    )
    axes[0, 1].set_title("Technology Counts in Refined Scenarios")
    axes[0, 1].set_ylabel("Number of scenarios")
    axes[0, 1].set_xlabel("Technology")
    axes[0, 1].tick_params(axis="x", rotation=35)
    axes[0, 1].grid(True, axis="y", alpha=0.3)
    for bar, count in zip(bars, technology_counts.values):
        axes[0, 1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(count),
            ha="center",
            va="bottom",
        )

    risks = [scenario.risk_score for scenario in collaborative_scenarios]
    npvs = [scenario.npv for scenario in collaborative_scenarios]
    scatter = axes[1, 0].scatter(
        risks,
        npvs,
        alpha=0.7,
        s=60,
        c=npvs,
        cmap="viridis",
        label="Refined scenario",
    )
    colorbar = fig.colorbar(scatter, ax=axes[1, 0])
    colorbar.set_label("NPV ($M)")
    axes[1, 0].scatter(
        final_scenario.risk_score,
        final_scenario.npv,
        s=200,
        c="red",
        marker="*",
        edgecolors="black",
        linewidth=1,
        label="Selected recommendation",
    )
    axes[1, 0].set_xlabel("Risk score (0–1; lower is preferred)")
    axes[1, 0].set_ylabel("NPV ($M)")
    axes[1, 0].set_title("Modeled Risk–NPV Trade-off")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    investments = [
        scenario.investment_cost for scenario in collaborative_scenarios
    ]
    axes[1, 1].hist(
        investments,
        bins=15,
        alpha=0.7,
        color="mediumpurple",
        edgecolor="black",
    )
    axes[1, 1].axvline(
        final_scenario.investment_cost,
        color="red",
        linestyle="--",
        linewidth=2,
        label="Selected recommendation",
    )
    axes[1, 1].axvline(
        constraints.max_investment,
        color="black",
        linestyle=":",
        linewidth=2,
        label="Stated investment limit",
    )
    axes[1, 1].set_xlabel("Investment cost ($M)")
    axes[1, 1].set_ylabel("Number of scenarios")
    axes[1, 1].set_title("Investment Costs of Refined Scenarios")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print("Plots summarize the simulated scenarios and stated model assumptions.")
else:
    print("Plots are unavailable because no refined recommendation is available.")

## Interpreting the plots

The NPV comparison displays each approved scenario before and after the fixed refinement rule; its distance from the no-change line follows from the rule rather than measured feedback. The technology and investment plots summarize only the refined shortlist, not every possible port expansion plan. The risk–NPV plot shows the trade-off represented by the model, and the highlighted point is the recommendation selected by the combined score. None of these plots represents a confidence interval or a forecast of real-world performance.

In [ ]:
# Summarize the selected scenario and the limits of the simulated comparison.
print("\n" + "=" * 80)
print("HUMAN–AI PARTNERSHIP: FINAL ANALYSIS")
print("=" * 80)

if final_scenario is not None and human_scenarios:
    pure_ai_best = max(ai_scenarios, key=lambda scenario: scenario.calculate_ai_fitness())
    npv_difference = final_scenario.npv - pure_ai_best.npv
    npv_difference_percent = (
        npv_difference / abs(pure_ai_best.npv) * 100
        if pure_ai_best.npv != 0
        else float("nan")
    )
    risk_difference_percentage_points = (
        pure_ai_best.risk_score - final_scenario.risk_score
    ) * 100

    print("\nSelected scenario")
    print(
        f"Expansion year: {final_scenario.expansion_year}; "
        f"capacity increase: {final_scenario.capacity_increase:.0%}; "
        f"technology: {final_scenario.technology.replace('_', ' ')}"
    )
    print(f"Investment: ${final_scenario.investment_cost:.1f}M")
    print(f"Investment limit: ${constraints.max_investment:.1f}M")
    print(
        "Budget check: "
        + (
            "within the stated limit"
            if final_scenario.investment_cost <= constraints.max_investment
            else "above the stated limit"
        )
    )
    print(f"Modeled NPV: ${final_scenario.npv:.1f}M")
    print(f"Modeled risk score: {final_scenario.risk_score:.1%}")
    print(
        "Simulated stakeholder acceptance: "
        f"{final_scenario.stakeholder_acceptance:.1%} "
        f"(threshold: {constraints.stakeholder_acceptance_threshold:.0%})"
    )

    print("\nDescriptive comparison with the highest AI-fitness candidate")
    print(f"AI-fitness candidate NPV: ${pure_ai_best.npv:.1f}M")
    print(f"NPV difference: ${npv_difference:.1f}M ({npv_difference_percent:.1f}%)")
    print(
        "Risk-score difference: "
        f"{risk_difference_percentage_points:.1f} percentage points"
    )
    print(
        "This is a descriptive comparison, not a causal estimate: the selected "
        "scenario comes from a screened shortlist and its NPV and risk include "
        "the fixed refinement adjustments."
    )

    print("\nInterpretation and practical considerations")
    print("- The recommendation depends on the encoded scenario options and scoring rules.")
    print("- Human-evaluation attributes are simulated rather than collected from experts.")
    print("- Environmental, labor, community-noise, and capacity-buffer effects are not calculated.")
    print("- A real planning process would require validated forecasts, cost estimates, and stakeholder input.")
    print("- The final score supports structured discussion; it does not replace expert review.")
else:
    print("Final analysis is unavailable because no scenario passed the evaluation stages.")

print("\nAnalysis of the simulated human–AI workflow is complete.")